<a href="https://colab.research.google.com/github/sarahayek98/Alzheimer-detection-using-3-machine-learning-algorithms-/blob/main/Random_forest_algorithms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

In [3]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define the path to your folder
folder_name = 'Alzheimer Dataset'
folder_path = f'/content/drive/MyDrive/{folder_name}'

# List files in the folder
files = os.listdir(folder_path)
print(files)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['test', 'train']


In [4]:
PATH = "/content/drive/MyDrive/Alzheimer Dataset/train"

BATCH_SIZE = 32
IMG_SIZE = 224

TRAIN_DS = tf.keras.utils.image_dataset_from_directory(
  PATH,
  validation_split=0.2,
  subset="training",
  seed=123,
  image_size=(IMG_SIZE, IMG_SIZE),
  batch_size=BATCH_SIZE)

VALID_DS = tf.keras.utils.image_dataset_from_directory(
  PATH,
  validation_split=0.2,
  subset="validation",
  seed=123,
  image_size=(IMG_SIZE, IMG_SIZE),
  batch_size=BATCH_SIZE)

CLASSES = TRAIN_DS.class_names

count = np.zeros(len(CLASSES), dtype=np.int32)
for _, labels in TRAIN_DS:
    y, _, c = tf.unique_with_counts(labels)
    count[y.numpy()] += c.numpy()
class_weight = dict(enumerate(count))

AUTOTUNE = tf.data.AUTOTUNE

TRAIN_DS = TRAIN_DS.cache().prefetch(buffer_size=AUTOTUNE)
VALID_DS = VALID_DS.cache().prefetch(buffer_size=AUTOTUNE)

Found 5121 files belonging to 4 classes.
Using 4097 files for training.
Found 5121 files belonging to 4 classes.
Using 1024 files for validation.


In [5]:
model = tf.keras.models.Sequential(name="DeepReLUwithEfficientNetB0")
model.add(tf.keras.applications.efficientnet.EfficientNetB0(include_top=False, weights='imagenet', input_shape=(224, 224, 3)))
model.add(tf.keras.layers.Flatten())
model.add(tf.keras.layers.Dense(4096, activation='relu'))
model.add(tf.keras.layers.Dense(1024, activation='relu'))
model.add(tf.keras.layers.Dense(256, activation='relu'))
model.add(tf.keras.layers.Dense(64, activation='relu'))
model.add(tf.keras.layers.Dense(len(CLASSES), activation='softmax'))
model.layers[0].trainable = False
model.summary()

Model: "DeepReLUwithEfficientNetB0"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 efficientnetb0 (Functional  (None, 7, 7, 1280)        4049571   
 )                                                               
                                                                 
 flatten (Flatten)           (None, 62720)             0         
                                                                 
 dense (Dense)               (None, 4096)              256905216 
                                                                 
 dense_1 (Dense)             (None, 1024)              4195328   
                                                                 
 dense_2 (Dense)             (None, 256)               262400    
                                                                 
 dense_3 (Dense)             (None, 64)                16448     
                                        

In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001), loss=tf.keras.losses.SparseCategoricalCrossentropy(), metrics=["accuracy"])
training_history = model.fit(TRAIN_DS, validation_data=VALID_DS, epochs=30, class_weight=class_weight)
results = pd.DataFrame({
    "Training Accuracy": training_history.history['accuracy'],
    "Validation Accuracy": training_history.history['val_accuracy'],
    "Training Loss": training_history.history['loss'],
    "Validation Loss": training_history.history['val_loss']
})

Epoch 1/30
 99/129 [======================>.......] - ETA: 3:54 - loss: 1484.8754 - accuracy: 0.5322

In [ ]:
model.layers[0].trainable = True
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001), loss=tf.keras.losses.SparseCategoricalCrossentropy(), metrics=["accuracy"])
training_history = model.fit(TRAIN_DS, validation_data=VALID_DS, epochs=15, class_weight=class_weight)
results_unfrozen = pd.DataFrame({
    "Training Accuracy": training_history.history['accuracy'],
    "Validation Accuracy": training_history.history['val_accuracy'],
    "Training Loss": training_history.history['loss'],
    "Validation Loss": training_history.history['val_loss']
})

In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001), loss=tf.keras.losses.SparseCategoricalCrossentropy(), metrics=["accuracy"])
training_history = model.fit(TRAIN_DS, validation_data=VALID_DS, epochs=15, class_weight=class_weight)
results_unfrozen_low_lr = pd.DataFrame({
    "Training Accuracy": training_history.history['accuracy'],
    "Validation Accuracy": training_history.history['val_accuracy'],
    "Training Loss": training_history.history['loss'],
    "Validation Loss": training_history.history['val_loss']
})

In [ ]:
training_history = model.fit(TRAIN_DS, validation_data=VALID_DS, epochs=30, class_weight=class_weight,
                             callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights= True, baseline=0.99)])

results_unfrozen_low_lr_final = pd.DataFrame({
    "Training Accuracy": training_history.history['accuracy'],
    "Validation Accuracy": training_history.history['val_accuracy'],
    "Training Loss": training_history.history['loss'],
    "Validation Loss": training_history.history['val_loss']
})

In [ ]:
import numpy as np
predictions = []
labels =  []
for x, y in VALID_DS:
    predictions.append(list(np.argmax(model.predict(x), axis = -1)))
    labels.append(list(y.numpy()))

predictions_final = []
true_labels = []
for l in predictions:
    predictions_final += l
for l in labels:
    true_labels += l

In [ ]:
print(CLASSES)
CLASS_NAMES = ['Mild\nDemented', 'Moderate\nDemented', 'Non\nDemented', 'Very Mild\nDemented']

In [ ]:
cm = confusion_matrix(true_labels, predictions_final)
ax = sns.heatmap(data=cm, annot=True, cmap='crest', fmt='g', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, linecolor='black', linewidth=0.5)
fig = ax.get_figure()
fig.savefig('confusion-matrix.pdf', bbox_inches='tight')

In [ ]:
print(classification_report(true_labels, predictions_final))

In [ ]:
sns.set_style("white")
final_results = pd.concat([results, results_unfrozen, results_unfrozen_low_lr, results_unfrozen_low_lr_final], ignore_index=True).reset_index()
final_results.columns = ["Epochs", "Training Accuracy", "Validation Accuracy", "Training Loss", "Validation Loss"]
final_results["Epochs"] += 1
melted_results = pd.melt(final_results, id_vars=['Epochs'], var_name='Metric', value_name='Value')

fig = plt.figure(figsize=(6,4))
fig.subplots_adjust(hspace=0.4, wspace=0.4)

ax = fig.add_subplot(2, 1, 1)
ax = sns.lineplot(data=melted_results[melted_results["Metric"].isin(["Training Accuracy", "Validation Accuracy"])], y="Value", x="Epochs", hue="Metric", linewidth = 2, ax=ax)
ax.set_xlabel("Epochs", fontdict={'weight': 'bold'})
ax.set_ylabel("Accuracy", fontdict={'weight': 'bold'})
ax.margins(x=0)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1.06))

ax = fig.add_subplot(2, 1, 2)
ax = sns.lineplot(data=melted_results[melted_results["Metric"].isin(["Training Loss", "Validation Loss"])], y="Value", x="Epochs", hue="Metric", linewidth = 2, ax=ax)
ax.set_xlabel("Epochs", fontdict={'weight': 'bold'})
ax.set_ylabel("Loss", fontdict={'weight': 'bold'})
ax.margins(x=0)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1.06))

In [ ]:
fig, ax = plt.subplots(figsize=(6,2.5))
ax = sns.lineplot(data=melted_results[melted_results["Metric"].isin(["Training Accuracy", "Validation Accuracy"])], y="Value", x="Epochs", hue="Metric", linewidth = 2, ax=ax)
ax.set_xlabel("Epochs", fontdict={'weight': 'bold'})
ax.set_ylabel("Accuracy", fontdict={'weight': 'bold'})
ax.margins(x=0)
sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1.06))
plt.vlines(x = 30, ymin = 0.5, ymax = 1.1, colors = 'purple', linestyles='dashed', linewidth=0.7)
plt.text(10, 0.55, "Frozen Base Model", verticalalignment='center', fontdict={'fontsize': 'x-small', 'fontweight': 'bold'})
plt.vlines(x = 45, ymin = 0.5, ymax = 1.1, colors = 'purple', linestyles='dashed', linewidth=0.7)
plt.text(37.5, 0.5, "Unfrozen\nBase Model", horizontalalignment='center', fontdict={'fontsize': 'x-small', 'fontweight': 'bold'})
plt.text(55, 0.5, "Reduced\nLearning\nRate", horizontalalignment='center', fontdict={'fontsize': 'x-small', 'fontweight': 'bold'})
plt.savefig('efficient-ad-training-accuracy.pdf', bbox_inches='tight')

In [ ]:
model.save("Efficient-AD.keras")

In [ ]:
IMG_HEIGHT = 128
IMG_WIDTH = 128
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
"/content/drive/MyDrive/Alzheimer Dataset/train",
seed=123,
image_size=(IMG_HEIGHT, IMG_WIDTH),
batch_size=64
)

test_ds = tf.keras.preprocessing.image_dataset_from_directory(
"/content/drive/MyDrive/Alzheimer Dataset/test",
seed=123,
image_size=(IMG_HEIGHT, IMG_WIDTH),
batch_size=64
)




In [ ]:
class_names = train_ds.class_names
print(class_names)
train_ds

In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")

In [ ]:
fig = plt.figure()
ax = fig.add_axes([0,0,1,1])
size = [896,64,3200,2240]
ax.bar(class_names,size)
plt.show

In [ ]:
import tensorflow as tf

# Analyze image properties
def analyze_images(images):
    for image in images:
        print(f"Image Size: {image.shape} - Image Type: {type(image)} - Pixel Value Range: {tf.reduce_min(image)} to {tf.reduce_max(image)}")

# Assuming images is a TensorFlow tensor
analyze_images(images[:5])


In [ ]:
# Normalize images
def preprocess_images(images):
    processed_images = []
    for image in images:
        # Normalize pixel values
        normalized_image = image / 255.0
        processed_images.append(normalized_image)
    return processed_images

processed_images = preprocess_images(images)
processed_images

Edge Detection: This can help in highlighting the outlines or shapes within the MRI scans. A common method for edge detection is using the Canny edge detector.

In [ ]:
import cv2

def apply_edge_detection(image):
    # Convert normalized image to 8-bit format
    image_8bit = np.uint8(image * 255)
    # Apply Canny edge detection
    edges = cv2.Canny(image_8bit, threshold1=100, threshold2=200)
    return edges

Image Segmentation:
This involves dividing the image into segments to make analysis easier. Since this can be complex, a simple approach is thresholding, where we create a binary image from the original one.

In [ ]:
def ensure_8bit_grayscale(image):
    # Convert normalized images to 8-bit
    if image.dtype != np.uint8:
        image = (image * 255).astype(np.uint8)

    # Convert color images to grayscale
    if len(image.shape) == 3 and image.shape[2] == 3:  # Check for 3 color channels
        image = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)

    return image

def apply_adaptive_thresholding(image):
    image_8bit = ensure_8bit_grayscale(image)
    return cv2.adaptiveThreshold(image_8bit, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)

Feature Extraction
Objective: Extract numerical features from the images that can be used by machine learning algorithms. Possible Features:

Texture Analysis: Texture features like contrast, entropy, etc., can be extracted. A simple way to start is to use the Gray Level Co-occurrence Matrix (GLCM).

In [ ]:
def extract_basic_texture_features(image):
    mean = np.mean(image)
    variance = np.var(image)
    std_dev = np.std(image)
    return mean, variance, std_dev

Data Augmentation
Objective: Increase the diversity of your dataset by applying various transformations to the images. This helps in improving the robustness of the model. Methods:

Rotating Images: Rotate images by various angles.

Flipping Images: Flip images horizontally or vertically.

Scaling: Zoom in or out of the images.

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2
)

Extract Basic Texture Features
We'll use your extract_basic_texture_features function to extract basic statistical measures from a subset of your preprocessed images.

In [ ]:
# Extract features from a subset of images
features = [extract_basic_texture_features(image) for image in processed_images[:10]]  # Adjust the subset size as needed

# Optionally, convert the features to a DataFrame for better visualization
import pandas as pd
features_df = pd.DataFrame(features, columns=['Mean', 'Variance', 'Standard Deviation'])
print(features_df)

In [ ]:
tf.experimental.numpy.experimental_enable_numpy_behavior()


Apply Image Transformations
We'll start with two simple transformations: edge detection and thresholding. These can help highlight different aspects of the MRI images

In [ ]:
import matplotlib.pyplot as plt

# Function to display normalized images
def display_normalized_images(images, num_images=5):
    plt.figure(figsize=(15, 3))
    for i in range(num_images):
        ax = plt.subplot(1, num_images, i + 1)
        plt.imshow(images[i], cmap='gray')  # Assuming images are grayscale
        plt.axis('off')
    plt.show()



In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

# Function to apply adaptive thresholding
def apply_adaptive_thresholding(image):
    # Ensure the image is in 8-bit grayscale format
    image = ensure_8bit_grayscale(image)

    # Normalize pixel values to the range [0, 255]
    image = cv2.normalize(image, None, 0, 255, cv2.NORM_MINMAX)

    # Your implementation of adaptive thresholding
    # Use cv2.THRESH_BINARY_INV to make the background white
    thresholded_image = cv2.adaptiveThreshold(image, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY_INV, blockSize=21, C=10)

    return thresholded_image

def ensure_8bit_grayscale(image):
    # Check if the image is already in the correct format
    if len(image.shape) == 2 and image.dtype == np.uint8:
        return image

    # Convert color images to grayscale
    if len(image.shape) == 3 and image.shape[2] == 3:  # Check for 3 color channels
        image = cv2.cvtColor(np.asarray(image), cv2.COLOR_RGB2GRAY)

    # Convert to 8-bit unsigned integer
    image = image.astype(np.uint8)

    return image

# Function to display original and thresholded images with histograms
def display_images_with_histograms(original_images, thresholded_images, num_images=5):
    plt.figure(figsize=(15, 6 * num_images))

    for i in range(num_images):
        # Original Image
        ax1 = plt.subplot(num_images, 2, 2 * i + 1)
        ax1.imshow(original_images[i], cmap='gray')
        ax1.set_title(f"Original Image {i + 1}")
        plt.axis('off')

        # Histogram of Original Image
        ax2 = plt.subplot(num_images, 2, 2 * i + 2)
        ax2.hist(original_images[i].ravel(), bins=256, color='black', alpha=0.7, rwidth=0.8)
        ax2.set_title(f"Histogram of Original Image {i + 1}")
        plt.xlabel('Pixel Value')
        plt.ylabel('Frequency')

    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(15, 6 * num_images))

    for i in range(num_images):
        # Thresholded Image
        ax1 = plt.subplot(num_images, 2, 2 * i + 1)
        ax1.imshow(thresholded_images[i], cmap='gray')
        ax1.set_title(f"Thresholded Image {i + 1}")
        plt.axis('off')

        # Histogram of Thresholded Image
        ax2 = plt.subplot(num_images, 2, 2 * i + 2)
        ax2.hist(thresholded_images[i].ravel(), bins=256, color='black', alpha=0.7, rwidth=0.8)
        ax2.set_title(f"Histogram of Thresholded Image {i + 1}")
        plt.xlabel('Pixel Value')
        plt.ylabel('Frequency')

    plt.tight_layout()
    plt.show()

# Directory containing your class subdirectories
class_directory = '/content/drive/MyDrive/Alzheimer Dataset/train'

# Get a list of all subdirectories in the class directory
class_subdirectories = [os.path.join(class_directory, subdir) for subdir in os.listdir(class_directory) if os.path.isdir(os.path.join(class_directory, subdir))]

# Read and preprocess images from each class
original_images = []
for class_subdir in class_subdirectories:
    class_image_paths = [os.path.join(class_subdir, filename) for filename in os.listdir(class_subdir)]
    class_images = [cv2.imread(path, cv2.IMREAD_GRAYSCALE) for path in class_image_paths if cv2.imread(path, cv2.IMREAD_GRAYSCALE) is not None]
    original_images.extend(class_images)

# Apply transformations to a subset of images
adaptive_thresholded_images = [apply_adaptive_thresholding(image) for image in original_images]

# Display the original and thresholded images with histograms
display_images_with_histograms(original_images, adaptive_thresholded_images, num_images=5)


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

# Function to apply adaptive thresholding
def apply_adaptive_thresholding(image):
    # Ensure the image is in 8-bit grayscale format
    image = ensure_8bit_grayscale(image)

    # Normalize pixel values to the range [0, 255]
    image = cv2.normalize(image, None, 0, 255, cv2.NORM_MINMAX)

    # Your implementation of adaptive thresholding
    # Use cv2.THRESH_BINARY_INV to make the background white
    thresholded_image = cv2.adaptiveThreshold(image, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY_INV, blockSize=21, C=10)

    return thresholded_image

def ensure_8bit_grayscale(image):
    # Check if the image is already in the correct format
    if len(image.shape) == 2 and image.dtype == np.uint8:
        return image

    # Convert color images to grayscale
    if len(image.shape) == 3 and image.shape[2] == 3:  # Check for 3 color channels
        image = cv2.cvtColor(np.asarray(image), cv2.COLOR_RGB2GRAY)

    # Convert to 8-bit unsigned integer
    image = image.astype(np.uint8)

    return image

# Function to display original and thresholded images
def display_images(original_images, thresholded_images, num_images=5):
    plt.figure(figsize=(15, 6 * num_images))

    for i in range(num_images):
        # Original Image
        ax1 = plt.subplot(num_images, 2, 2 * i + 1)
        ax1.imshow(original_images[i], cmap='gray')
        ax1.set_title(f"Original Image {i + 1}")
        plt.axis('off')

        # Thresholded Image
        ax2 = plt.subplot(num_images, 2, 2 * i + 2)
        ax2.imshow(thresholded_images[i], cmap='gray')
        ax2.set_title(f"Thresholded Image {i + 1}")
        plt.axis('off')

    plt.tight_layout()
    plt.show()

# Directory containing your class subdirectories
class_directory = '/content/drive/MyDrive/Alzheimer Dataset/train'

# Get a list of all subdirectories in the class directory
class_subdirectories = [os.path.join(class_directory, subdir) for subdir in os.listdir(class_directory) if os.path.isdir(os.path.join(class_directory, subdir))]

# Read and preprocess images from each class
original_images = []
for class_subdir in class_subdirectories:
    class_image_paths = [os.path.join(class_subdir, filename) for filename in os.listdir(class_subdir)]
    class_images = [cv2.imread(path, cv2.IMREAD_GRAYSCALE) for path in class_image_paths if cv2.imread(path, cv2.IMREAD_GRAYSCALE) is not None]
    original_images.extend(class_images)

# Apply transformations to a subset of images
adaptive_thresholded_images = [apply_adaptive_thresholding(image) for image in original_images]

# Display the original and thresholded images
display_images(original_images, adaptive_thresholded_images, num_images=5)


Applying data augmentaiton

In [ ]:
# Apply data augmentation to a subset of images
augmented_images = next(datagen.flow(np.array(processed_images[:10]), batch_size=10))

# Display augmented images
display_normalized_images(augmented_images, num_images=5)

In [ ]:
!pip install tensorflow


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix

# Function to train and plot accuracy
def train_and_plot(model, X_train, y_train, X_val, y_val, epochs=50):

    history = model.fit(X_train, y_train, epochs=epochs, validation_data=(X_val, y_val), verbose=2)
    # Calculate and plot confusion matrix
    y_pred = np.argmax(model.predict(X_val), axis=1)
    y_true = np.argmax(y_val, axis=1)
    cm = confusion_matrix(y_true, y_pred)
    plot_confusion_matrix(cm, classes=['MILD_DEMENTED', 'MODERATE_DEMENTED', 'NON_DEMENTED', 'VERY_MILD_DEMENTED'])
    plt.show()

    # Plot training history
    plot_training_history(history)


def plot_training_history(history):
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()

    plt.tight_layout()
    plt.savefig('history_plot.png')
    plt.show()


    plt.tight_layout()


def plot_confusion_matrix(cm, classes, figsize=(8, 6)):
    plt.figure(figsize=figsize)
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title('Confusion Matrix')
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()


In [ ]:
# Input shape
input_shape = ( 150, 150, 3)

Random Forest Model

In [ ]:
# Import necessary libraries
import numpy as np
import os
import cv2
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, confusion_matrix , classification_report


In [ ]:
train_folder = '/content/drive/MyDrive/Alzheimer Dataset/train'
test_folder = '/content/drive/MyDrive/Alzheimer Dataset/test'

In [ ]:
def preprocess_image(image_path, target_size=(64, 64)):
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)  # Convert to grayscale
    image = cv2.resize(image, target_size)  # Resize image
    image = image / 255.0  # Normalize pixel values
    return image

In [ ]:
def load_images_from_folder(folder):
    images = []
    labels = []
    for subfolder in os.listdir(folder):
        subfolder_path = os.path.join(folder, subfolder)
        for filename in os.listdir(subfolder_path):
            img_path = os.path.join(subfolder_path, filename)
            image = preprocess_image(img_path)
            images.append(image)
            labels.append(subfolder)
    return np.array(images), np.array(labels)

In [ ]:
X_train, y_train = load_images_from_folder(train_folder)
X_test, y_test = load_images_from_folder(test_folder)

In [ ]:
#Model Training
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
X_train_flattened = X_train.reshape(X_train.shape[0], -1)  # Flatten the image arrays
rf_classifier.fit(X_train_flattened, y_train)

In [ ]:
#  Model Evaluation
X_test_flattened = X_test.reshape(X_test.shape[0], -1)  # Flatten the image arrays
y_pred = rf_classifier.predict(X_test_flattened)
accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')


In [ ]:
print("Accuracy:", accuracy)
print("Recall:", recall)
print("F1 Score:", f1)

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, cmap='Blues', fmt='g', xticklabels=np.unique(y_test), yticklabels=np.unique(y_test))
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()
